# Теория. Процессы в Python

### 1. Что такое процесс?
**Процесс** — это работающий экземпляр программы. Когда вы запускаете свой скрипт, операционная система создает для него процесс.

Представьте, что вы запустили два разных браузера или две разные игры — это разные процессы. Внутри Python-скрипта мы можем программно создавать такие же независимые процессы для параллельной работы.

**Основные свойства процесса:**
*   **PID (Process ID):** Уникальный номер, по которому ОС находит вашу программу.
*   **Собственный интерпретатор:** У каждого процесса Python свой экземпляр интерпретатора и свой **GIL** (Global Interpreter Lock), поэтому они не мешают друг другу работать одновременно.

### 2. Изоляция памяти (Главное отличие)
У каждого процесса **своя собственная память**.
*   Если в основном процессе есть переменная `x = 10`, а дочерний процесс её изменит на `x = 20`, то в основном процессе она **останется равной 10**.
*   Процессы «не видят» данные друг друга. Это делает их безопасными (один не сломает данные другого), но затрудняет обмен информацией (для передачи данных нужно использовать специальные инструменты: очереди или каналы).

### 3. Зачем нужны процессы? (CPU-bound задачи)
В Python процессы используются для обхода **GIL**. Поскольку каждый процесс — это независимая единица, операционная система может распределить их по разным **ядрам процессора**.

**Когда использовать процессы:**
Когда ваша задача сильно нагружает «железо» (процессор):
*   **Математические вычисления:** Расчет сложных формул, матриц.
*   **Обработка данных:** Сжатие файлов, парсинг огромных массивов текста.
*   **Медиа:** Обработка изображений или видео.
*   **Машинное обучение:** Обучение моделей или предобработка признаков.

**Когда НЕ использовать:**
Если программа просто ждет ответа от сервера или загрузки файла из интернета (I/O-bound задачи) — в этом случае процессы создадут лишнюю нагрузку на систему, и лучше использовать потоки (threading).


In [16]:
# Подключаем библиотеки
import multiprocessing
import threading
import os
import time

In [23]:
# Глобальный список данных (в основном процессе)
Global_data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [18]:
print(f"ID основного процесса: {os.getpid()}")
print(f"Исходные данные: {Global_data}")

ID основного процесса: 6079
Исходные данные: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [20]:
#Создаем функцию, которую будем запускать
def worker_task(limit):
    current_pid = os.getpid()
    print(f"[/] Дочерний процесс {current_pid} начал работу.")

    # Считаем сумму первых N элементов
    results = 0
    for i in range(limit):
        results += Global_data[i]

    # Добавляем результат в список (изменяем "свою" копию Global_data)
    Global_data.append(results)

    print(f"[/] Дочерний процесс {current_pid}: Сумма={results}. Список теперь: {Global_data}")

In [21]:
# Создаем процесс, который просуммирует первые 5 элементов
p = multiprocessing.Process(target=worker_task, args=(5,))

print("--- Старт процесса ---")
p.start()
p.join()
print("--- Процесс завершен ---")

print(f"Основной процесс (PID {os.getpid()}). Список после работы процесса: {Global_data}")

--- Старт процесса ---
[/] Дочерний процесс 27551 начал работу.
[/] Дочерний процесс 27551: Сумма=15. Список теперь: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15]
--- Процесс завершен ---
Основной процесс (PID 6079). Список после работы процесса: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [22]:
print(f"Исходный список перед потоком: {Global_data}")

# Запускаем ту же функцию, но в потоке
t = threading.Thread(target=worker_task, args=(3,))

print("--- Старт потока ---")
t.start()
t.join()
print("--- Поток завершен ---")

print(f"Основной процесс. Список после работы потока: {Global_data}")

Исходный список перед потоком: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
--- Старт потока ---
[/] Дочерний процесс 6079 начал работу.
[/] Дочерний процесс 6079: Сумма=6. Список теперь: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 6]
--- Поток завершен ---
Основной процесс. Список после работы потока: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 6]


In [29]:
#Гибридная модель (Процесс + Потоки внутри)
def process_with_threads(limit):
    pid = os.getpid()
    print(f"Процесс {pid} запущен. Внутри него создаем потоки...")

    # Создаем два потока, которые делают ту же задачу
    threads = []
    for i in range(2):
        t = threading.Thread(target=worker_task, args=(i+2,))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()
    print(f"Процесс {pid} завершил внутренние потоки. Его список: {Global_data}")

In [34]:
    p_hybrid = multiprocessing.Process(target=process_with_threads, args=(2,))
    p_hybrid.start()
    p_hybrid.join()

Процесс 28515 запущен. Внутри него создаем потоки...
[/] Дочерний процесс 28515 начал работу.[/] Дочерний процесс 28515 начал работу.

[/] Дочерний процесс 28515: Сумма=3. Список теперь: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 3]
[/] Дочерний процесс 28515: Сумма=6. Список теперь: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 3, 6]
Процесс 28515 завершил внутренние потоки. Его список: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 3, 6]


In [54]:
def worker_return(limit, queue):
    current_pid = os.getpid()

    # 1. Работа с локальной копией памяти
    partial_sum = sum(Global_data[:limit])

    # Пытаемся изменить "глобальный" список
    Global_data.append(f"SUM:{partial_sum}")

    print(f"[/] Дочерний процесс {current_pid}:")
    print(f"    - Локальный Global_data: {Global_data}")
    print(f"    - Посчитанная сумма: {partial_sum}")

    # 2. Передача данных наружу через Queue
    queue.put(partial_sum)

    # 3. Пытаемся вернуть значение через return (оно пропадет)
    return partial_sum

In [55]:
# Создаем очередь для получения данных
results_queue = multiprocessing.Queue()

print(f"[*] Основной процесс (PID {os.getpid()}):")
print(f"    - Исходный Global_data: {Global_data}")

# Запускаем процесс
p = multiprocessing.Process(target=worker_return, args=(3, results_queue))
p.start()

# Пытаемся получить результат через return (join возвращает None)
return_val = p.join()

# Получаем результат через Queue
if not results_queue.empty():
    queue_val = results_queue.get()
else:
    queue_val = "Ничего не пришло"

print(f"\n[*] Итоги в основном процессе:")
print(f"    - Global_data остался прежним: {Global_data}")
print(f"    - Результат через return: {return_val}")
print(f"    - Результат через Queue: {queue_val}")

[*] Основной процесс (PID 6079):
    - Исходный Global_data: [1, 2, 3, 4, 5]
[/] Дочерний процесс 34759:
    - Локальный Global_data: [1, 2, 3, 4, 5, 'SUM:6']
    - Посчитанная сумма: 6

[*] Итоги в основном процессе:
    - Global_data остался прежним: [1, 2, 3, 4, 5]
    - Результат через return: None
    - Результат через Queue: 6


# Теория — Пул процессов (multiprocessing.Pool)

### Что такое Pool?
Представьте, что у вас есть 1000 задач. Создавать 1000 процессов вручную — плохая идея (система просто зависнет от нагрузки).
**Pool (Пул)** — это диспетчер, который:
1. Создает фиксированное количество рабочих процессов (обычно по числу ядер вашего CPU).
2. Сам распределяет задачи между ними.
3. Когда один процесс заканчивает работу, Pool тут же дает ему следующую задачу.
4. **Автоматически собирает результаты** всех процессов и возвращает их вам в один список.



In [36]:
def sum_progression(start, end, step=1):
    """Считает сумму части прогрессии"""
    partial_sum = sum(range(start, end, step))
    print(f"[Process {os.getpid()}] Посчитана сумма от {start} до {end}: {partial_sum}")
    return partial_sum

In [37]:
# РУЧНОЕ РАЗДЕЛЕНИЕ ЗАДАЧИ
# Первый процесс считает от 0 до 50
# Второй процесс считает от 50 до 100

p1 = multiprocessing.Process(target=sum_progression, args=(0, 50))
p2 = multiprocessing.Process(target=sum_progression, args=(50, 101))

p1.start()
p2.start()

p1.join()
p2.join()

[Process 29683] Посчитана сумма от 0 до 50: 1225
[Process 29686] Посчитана сумма от 50 до 101: 3825


In [38]:
tasks = [(0, 25), (25, 50), (50, 75), (75, 101)]

print(f"--- Запуск Pool (количество ядер: {multiprocessing.cpu_count()}) ---")

# Создаем пул процессов
with multiprocessing.Pool(processes=4) as pool:
    # Используем starmap, так как у функции несколько аргументов
    # Он сам распределит задачи из списка tasks по процессам
    results = pool.starmap(sum_progression, tasks)

print(f"--- Pool завершил работу ---")
print(f"Результаты по частям: {results}")
print(f"Итоговая сумма: {sum(results)}")

--- Запуск Pool (количество ядер: 2) ---
[Process 29798] Посчитана сумма от 25 до 50: 925[Process 29799] Посчитана сумма от 50 до 75: 1550[Process 29797] Посчитана сумма от 0 до 25: 300[Process 29800] Посчитана сумма от 75 до 101: 2275



--- Pool завершил работу ---
Результаты по частям: [300, 925, 1550, 2275]
Итоговая сумма: 5050


# Практическое занятие: Параллельное обучение нейросетей

**Цель работы:** Сравнить три архитектуры параллельного выполнения задач в Python и наглядно увидеть влияние GIL и изоляции процессов на скорость и результат.

### Описание задачи:
Вам необходимо обучить 9 нейронных сетей с разными гиперпараметрами. Для этого мы разделим их на 3 группы по 3 сети. Каждая группа имеет свои особенности (разные нейроны, разные функции активации или разная глубина).

### Задание:
Реализуйте и замерьте время выполнения для трех сценариев:
1.  **Вариант «Максимальный параллелизм»:** Запуск 9 независимых процессов (по одному на каждую сеть).
2.  **Вариант «Групповая обработка»:** Запуск 3 процессов, каждый из которых по очереди (последовательно) обучает 3 сети из своей группы.
3.  **Вариант «Гибридная модель»:** Запуск 3 процессов, каждый из которых запускает внутри себя 3 потока для обучения сетей.


In [1]:
#импорт библиотек
import os
import time
import multiprocessing
import threading
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
import numpy as np
print(f"Количество ядер в системе: {os.cpu_count()}")

Количество ядер в системе: 32


In [2]:
# 1. Генерация данных
print("Подготовка данных...")
X, y = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Подготовка данных...


In [3]:
# Конфигурации сетей (9 штук)
settings = [
    # Группа 1: Разные нейроны
    {'params': [(100,), 'relu'], 'group': 'Neurons'},
    {'params': [(200,), 'relu'], 'group': 'Neurons'},
    {'params': [(300,), 'relu'], 'group': 'Neurons'},
    # Группа 2: Разные активации
    {'params': [(150,), 'relu'], 'group': 'Activations'},
    {'params': [(150,), 'tanh'], 'group': 'Activations'},
    {'params': [(150,), 'logistic'], 'group': 'Activations'},
    # Группа 3: Разные слои
    {'params': [(50,), 'relu'], 'group': 'Layers'},
    {'params': [(50, 50), 'relu'], 'group': 'Layers'},
    {'params': [(50, 50, 50), 'relu'], 'group': 'Layers'}
]

In [4]:
def train_task(index, queue):
    # Извлекаем параметры из словаря
    layers, activation = settings[index]['params']

    model = MLPRegressor(hidden_layer_sizes=layers, activation=activation, max_iter=400, random_state=42)
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)

    # Результат кладем в очередь (т.к. return в процессах не работает)
    queue.put((index, prediction))


In [5]:
# 4. Воркер для последовательного обучения внутри процесса
def worker_sequential(indices, queue):
    for i in indices:
        train_task(i, queue)

In [6]:
# 5. Воркер для гибридного обучения (потоки внутри процесса)
def worker_hybrid(indices, queue):
    threads = []
    for i in indices:
        t = threading.Thread(target=train_task, args=(i, queue))
        threads.append(t)
        t.start()
    for t in threads:
        t.join()

In [7]:
# Разделение на 3 группы по индексам
groups = [(0, 1, 2), (3, 4, 5), (6, 7, 8)]

# --- ЭКСПЕРИМЕНТ 1: 9 НЕЗАВИСИМЫХ ПРОЦЕССОВ ---
print("\n--- Эксперимент 1: 9 независимых процессов ---")
q1 = multiprocessing.Queue()
start = time.time()
procs1 = [multiprocessing.Process(target=train_task, args=(i, q1)) for i in range(9)]
for p in procs1: p.start()
for p in procs1: p.join()
time1 = time.time() - start


--- Эксперимент 1: 9 независимых процессов ---


In [8]:
# --- ЭКСПЕРИМЕНТ 2: 3 ПРОЦЕССА (ПОСЛЕДОВАТЕЛЬНО ВНУТРИ) ---
print("--- Эксперимент 2: 3 процесса (по 3 сети по очереди) ---")
q2 = multiprocessing.Queue()
start = time.time()
procs2 = [multiprocessing.Process(target=worker_sequential, args=(g, q2)) for g in groups]
for p in procs2: p.start()
for p in procs2: p.join()
time2 = time.time() - start

--- Эксперимент 2: 3 процесса (по 3 сети по очереди) ---


In [9]:
# --- ЭКСПЕРИМЕНТ 3: 3 ПРОЦЕССА (ПО 3 ПОТОКА ВНУТРИ) ---
print("--- Эксперимент 3: 3 процесса (в каждом по 3 потока) ---")
q3 = multiprocessing.Queue()
start = time.time()
procs3 = [multiprocessing.Process(target=worker_hybrid, args=(g, q3)) for g in groups]
for p in procs3: p.start()
for p in procs3: p.join()
time3 = time.time() - start

--- Эксперимент 3: 3 процесса (в каждом по 3 потока) ---


In [10]:
print("\n" + "="*45)
print(f"Вариант 1 (9 процессов): {time1:.2f} сек")
print(f"Вариант 2 (3 проц. послед.): {time2:.2f} сек")
print(f"Вариант 3 (3 проц. гибрид): {time3:.2f} сек")
print("="*45)


Вариант 1 (9 процессов): 0.07 сек
Вариант 2 (3 проц. послед.): 0.05 сек
Вариант 3 (3 проц. гибрид): 0.05 сек
